# NativeStudio — Qwen3-4B LoRA fine-tune (Colab)

Trains a LoRA adapter for `nativestudio:coder` (base: Qwen3-4B) on the three focus areas:
1. Casual conversation warmth
2. Tool-calling reliability
3. NativeStudio/project-specific coding style

**Before running:** Runtime → Change runtime type → T4 GPU (free tier).

**You'll need:** `train.jsonl` and `valid.jsonl` from `training/data/` in the project — generate/refresh them locally with `python3 training/build_dataset.py`, then upload both when prompted in the cell below.

In [ ]:
%%capture
!pip install unsloth
!pip install --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 1536   # matches training/data's actual max token length (~1416); raise if you add longer examples
dtype = None             # auto-detect (bfloat16 on T4/A100)
load_in_4bit = True      # QLoRA — keeps VRAM usage low

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-4B",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

In [ ]:
# LoRA config — rank 8 matches what we used locally; feel free to raise to 16
# for more capacity if the bigger dataset still underfits.
model = FastLanguageModel.get_peft_model(
    model,
    r = 8,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

## Upload the dataset
Run the next cell, then pick both `train.jsonl` and `valid.jsonl` from `training/data/` when the file picker appears.

In [ ]:
from google.colab import files
uploaded = files.upload()  # select train.jsonl and valid.jsonl together
assert "train.jsonl" in uploaded and "valid.jsonl" in uploaded, "Upload both train.jsonl and valid.jsonl"

In [ ]:
import json
from datasets import Dataset

def load_rows(path):
    rows = []
    with open(path) as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

def to_text(row):
    # Renders the same way Ollama's Qwen3 template would, including <tool_call> tags,
    # so the model trains on exactly the format it'll see at inference time.
    text = tokenizer.apply_chat_template(
        row["messages"],
        tools = row.get("tools"),
        tokenize = False,
        add_generation_prompt = False,
    )
    return {"text": text}

train_rows = load_rows("train.jsonl")
valid_rows = load_rows("valid.jsonl")

train_dataset = Dataset.from_list(train_rows).map(to_text, remove_columns=["messages"] + (["tools"] if "tools" in train_rows[0] else []))
valid_dataset = Dataset.from_list(valid_rows).map(to_text, remove_columns=["messages"] + (["tools"] if "tools" in valid_rows[0] else []))

print(f"{len(train_dataset)} train / {len(valid_dataset)} valid examples")
print(train_dataset[0]["text"][:500])

In [ ]:
from trl import SFTTrainer, SFTConfig

# num_train_epochs scaled for a dataset this size — watch train_loss vs
# eval_loss in the output below the same way we did locally: if eval_loss
# plateaus while train_loss keeps dropping, stop early (reduce epochs and
# rerun) rather than trusting the final checkpoint by default.
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = valid_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        num_train_epochs = 4,
        learning_rate = 2e-4,
        logging_steps = 5,
        eval_strategy = "epoch",
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

trainer_stats = trainer.train()

## Save and download the adapter
Check the eval_loss trend printed above first — if it plateaued early (like our local run did), consider re-running with fewer epochs before trusting this checkpoint.

In [ ]:
model.save_pretrained("lora_adapter")
tokenizer.save_pretrained("lora_adapter")

!zip -r lora_adapter.zip lora_adapter
files.download("lora_adapter.zip")

## Optional: export straight to GGUF (skips the whole fuse/convert dance we did locally)
Unsloth's own GGUF exporter supports Qwen3 directly (unlike `mlx_lm.fuse --export-gguf`, which doesn't yet). This produces a ready-to-import file for `ollama create`.

In [ ]:
model.save_pretrained_gguf("nativestudio-coder-lora-gguf", tokenizer, quantization_method = "q4_k_m")
!zip nativestudio-coder-lora.gguf.zip nativestudio-coder-lora-gguf_gguf/*.gguf
files.download("nativestudio-coder-lora.gguf.zip")